## Linear operators
Import libraries

In [ ]:
import os
import sys
import json
import numpy as np
import scipy as sp
import tensorflow as tf
sys.path.append("/home/zanardi/Codes/ML/PyCOMET/pycomet/")

from tqdm import tqdm
from pycomet.networks.chemkin.plasma_cr.meno.blocks import PlasmaCROperators

In [ ]:
import utils

Define inputs

In [ ]:
# Paths
path_to_datagen = "./../00.datagen.r02/"
paths = {
  "dtb": path_to_datagen + "/database/",
  "data": path_to_datagen + "/data/test/",
  "saving": "./output/lin_ops/"
}
species = ("em", "Ar", "Arp")
os.makedirs(paths["saving"], exist_ok=True)
# deltaor
# > Epsilon value
eps = 1e-10
# > Confidence interval
alpha = 0.95
uncert_name = f"ci_{int(100*alpha)}" if (alpha > 0) else "std"

In [ ]:
# Time grid
tlim = [1e-8, 1e-6]
tdim = 200
t = np.geomspace(*tlim, num=tdim)

Define data indices

In [ ]:
# Data indices
with open(path_to_datagen + "/data/indices.json") as file:
  indices = json.load(file)

Load data and model

In [ ]:
data = utils.load_test_cases(paths["data"])

Initialize the linear operators constructor

In [ ]:
linops = PlasmaCROperators(
  species={k: paths["dtb"] + "/species/" + k + ".json" for k in species},
  rates=paths["dtb"] + "/rates/rates.hdf5",
  use_corr=False,
  scale_corr=False
)

In [ ]:
def compute_linops(x):
  n_em, n_ar1, Th, Te = x
  return linops((n_em, n_ar1, Th, Te, None), training=False)

Compute linear operator variability

In [ ]:
def compute_delta(t, y, teval=None):
  t = t.reshape(-1)
  axes = tuple(np.arange(y.ndim)[:-1].tolist())
  dydt = np.gradient(y, t, axis=-1, edge_order=2)
  delta = np.linalg.norm(dydt, axis=axes)
  delta /= np.linalg.norm(y/t, axis=axes)
  if (teval is not None):
    delta = sp.interpolate.interp1d(t, delta, kind="cubic")(teval)
  return 100 * delta

In [ ]:
deltas = {"A": []}
qoi = ("t", "n_em", "n_Ar1", "Th", "Te")
for (icase, idata) in tqdm(data.items()):
  idata = idata.to_numpy()
  ti, *xi = [np.take(idata, indices[q], axis=1) for q in qoi]
  Ai = tf.nest.map_structure(lambda x: x.numpy(), compute_linops(xi))
  deltas["A"].append(compute_delta(ti, Ai.T, t))
deltas = {k: np.vstack(d) for (k, d) in deltas.items()}
deltas = tf.nest.map_structure(
  lambda x: utils.compute_stats(x, axis=0, alpha=alpha),
  deltas
)

In [ ]:
for k in ("A",):
  utils.plot_err_continuous(
    x=t,
    mean=deltas[k][0],
    uncert=deltas[k][1],
    alpha=alpha,
    labels=[r"$t$ [s]", r"$\Delta\mathbf{%s}$" % (k,) + " [\%]"],
    scales=["log", "linear"],
    legend_loc="best",
    figname=paths["saving"]+ f"/delta_{k}.pdf",
    save=True,
    show=False
  )